<a id="top"></a>

# Lab L1305: Pass rates and regressions

```yaml
title: "Lab L1305: Pass rates and regressions"
keywords: evaluation, pass rate, samples, non-determinism, regression, compare, model comparison, Langfuse, experiment, run_experiment, lab
estimated duration: 30
```

> **Lesson:** L13. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L13/objectives.md).
> Problems 1–3 run **offline** — no API key needed (a deterministic model simulator). Problem 4 is a **tooled** bonus: it pushes a run to the cohort Langfuse, so it soft-skips unless *both* a model key and Langfuse are configured.

**Learning objectives.** By the end of this lab you can:

- **move from a verdict to a pass rate** — run a case K times and read how often it passes;
- **flag regressions** between two versions with `compare(...)` — pass→fail is the thing to catch;
- **push a run to Langfuse** — run the shared dataset as an experiment and read the pass rate in the UI.

> Solutions: `L1305_lab_solutions.ipynb`. Problems 1–3 need no API key; Problem 4 is a gated Langfuse bonus.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — a pass rate, not a verdict](#2-problem-1--a-pass-rate-not-a-verdict)
- [3. Problem 2 — flag the regressions](#3-problem-2--flag-the-regressions)
- [4. Problem 3 — read the instrument](#4-problem-3--read-the-instrument)
- [5. Problem 4 — push a run to Langfuse](#5-problem-4--push-a-run-to-langfuse)
- [6. Self-check](#6-self-check)

## 1. Setup

Run this cell once. The agent graph is non-deterministic, so we provide a **deterministic simulator** of it: `ModelSim(strong=...)` is a `run_case` that behaves like a stronger or weaker model. A strong model recovers from a tool error and looks a city up once; a weak model gives up on recovery and *intermittently runs away* on the lookup — so its pass rate is a fraction, reproducibly. You'll run the eval set and read the numbers.

In [ ]:
from typing import Any

from fluffy_potato_curriculum.common.agent_graph import RunResult, run
from fluffy_potato_curriculum.common.evals import (
    EvalCase,
    EvalResult,
    Scorer,
    compare,
    evaluate,
    tool_calls,
)
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS


def answer_correct(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Outcome scorer (loosened): is the reference answer in the final text?"""

    def norm(text: str) -> str:
        return text.replace(",", "").replace(" ", "")

    expected = str(example.reference_outputs["answer"])
    return EvalResult(key="answer_correct", score=norm(expected) in norm(run.final_text))


def no_runaway(*, run: RunResult, example: EvalCase) -> EvalResult:
    """Trajectory scorer: no repeated (tool, args) call and didn't hit the step cap."""
    signatures = [(name, tuple(sorted(args.items()))) for name, args in tool_calls(run)]
    repeated = len(signatures) != len(set(signatures))
    return EvalResult(key="no_runaway", score=not (repeated or run.termination == "max_steps"))


class ModelSim:
    """A deterministic run_case simulating a model of a given strength."""

    def __init__(self, *, strong: bool) -> None:
        self.strong = strong
        self._seen: dict[str, int] = {}

    def _attempt(self, case_id: str) -> int:
        n = self._seen.get(case_id, 0)
        self._seen[case_id] = n + 1
        return n

    def __call__(self, case: EvalCase) -> RunResult:
        n = self._attempt(case.id)
        if case.id == "recover" and self.strong:
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                tool_reply(tool_call("c2", "flaky_fetch", {"url": "https://ok"})),
                text_reply("The answer is 42."),
            ]
        elif case.id == "recover":
            script = [
                tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
                text_reply("Sorry, I could not fetch that URL."),
            ]
        elif case.id == "hard_lookup" and not self.strong and n % 3 == 1:
            # runaway: distinct call ids (r0, r1, ...) keep the graph's add_messages
            # reducer from de-duping the repeats, so it loops to the max_steps cap.
            script = [
                tool_reply(tool_call(f"r{i}", "lookup", {"city": "Atlantis"})) for i in range(8)
            ]
        elif case.id == "hard_lookup":
            script = [
                tool_reply(tool_call("c1", "lookup", {"city": "Tokyo"})),
                text_reply("Tokyo has a population of 37000000."),
            ]
        else:
            raise KeyError(case.id)
        return run(
            FakeModel(script), TOOLS, case.inputs["task"], max_steps=case.inputs.get("max_steps", 8)
        )


recover_case = EvalCase(
    id="recover",
    inputs={"task": "Fetch from https://crash; if it fails try https://ok."},
    reference_outputs={"answer": "42"},
)
hard_lookup_case = EvalCase(
    id="hard_lookup",
    inputs={"task": "What is the population of Tokyo?", "max_steps": 4},
    reference_outputs={"answer": "37000000"},
)
cases = [recover_case, hard_lookup_case]
scorers = [answer_correct, no_runaway]
print("setup ready:", [c.id for c in cases])

[↑ Back to top](#top)

## 2. Problem 1 — a pass rate, not a verdict

A single run can be luck. Run the **weak** model on the lookup case `samples=5` times and read the pass *rate* for the `no_runaway` check. Because the weak model intermittently runs away, the rate is a **fraction** — that fraction is the finding.

Fill in the `evaluate(...)` call (set `samples=5`) and read the rate with `report.pass_rate(case_id, key)`.

In [ ]:
weak = ModelSim(strong=False)
# TODO: run the eval set on just [hard_lookup_case] with samples=5.
report = ...  # TODO: evaluate(weak, [hard_lookup_case], scorers, samples=5)
print(report.table())

# TODO: read the no_runaway pass rate for "hard_lookup".
rate = ...  # TODO: report.pass_rate("hard_lookup", "no_runaway")
print("no_runaway pass rate:", rate)

# A flaky case: passes sometimes, fails sometimes -> strictly between 0 and 1.
assert 0.0 < rate < 1.0

[↑ Back to top](#top)

## 3. Problem 2 — flag the regressions

Now use the eval set as a **ratchet**. Build a report for the strong model (the baseline) and one for the weak model (the "after"), then call `compare(before, after)` to flag which `(case, scorer)` pairs went **pass → fail**.

Fill in the two `evaluate(...)` calls and the `compare(...)` call. The weak model should regress recovery and the lookup.

In [ ]:
strong_report = ...  # TODO: evaluate(ModelSim(strong=True), cases, scorers, samples=5)
weak_report = ...  # TODO: evaluate(ModelSim(strong=False), cases, scorers, samples=5)

delta = ...  # TODO: compare(strong_report, weak_report)
print("regressions (pass -> fail):", delta.regressions)
print("fixes       (fail -> pass):", delta.fixes)

# Recovery should be a clear regression for the weaker model.
assert ("recover", "answer_correct") in delta.regressions
assert delta.has_regressions

[↑ Back to top](#top)

## 4. Problem 3 — read the instrument

The eval set is a **measurement instrument**, not just a guard. Print the strong and weak tables side by side, then answer the written question.

In [ ]:
print("=== strong model ===")
print(evaluate(ModelSim(strong=True), cases, scorers, samples=5).table())
print("\n=== weak model ===")
print(evaluate(ModelSim(strong=False), cases, scorers, samples=5).table())

**TODO (written):** A teammate runs each case **once** and reports "the weak model passed recovery." Using what you saw above, explain in one or two sentences why that single run can't be trusted, and what number you'd report instead. Edit this cell.

_TODO_

[↑ Back to top](#top)

## 5. Problem 4 — push a run to Langfuse

You read pass rates off a printed table above. Now put them where the team can see them: run the **shared `l13-agent-evals` dataset** (uploaded by the `L1302` lecture) as a **Langfuse experiment** and read the pass rate in the UI.

The adapter and task are written for you — `as_evaluator(...)` turns each course scorer into a Langfuse evaluator, and `make_task(model_id)` runs the graph with a real model. **Your job:** call `dataset.run_experiment(...)` with them, then open the printed run URL and find the `answer_correct` / `no_runaway` pass rates.

This cell is **gated** on both `ANTHROPIC_API_KEY` and `langfuse_configured()`; offline it just prints a skip note, so the lab still runs top-to-bottom without keys.

In [ ]:
# Problem 4 (tooled): push a real run to Langfuse. Instead of reading the pass rate
# off a printed table, run the SHARED `l13-agent-evals` dataset (from the lecture) as
# a Langfuse EXPERIMENT and read the rate in the UI. Gated on BOTH a model key and a
# live Langfuse, so an offline run just skips.
from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_langfuse,
)


def as_evaluator(scorer: Scorer):
    """Your course `Scorer` IS a Langfuse evaluator, via a one-line adapter (given).

    A Langfuse evaluator is called `(*, input, output, expected_output, ...)`; our
    `output` is the `RunResult` the task returned, so the scorer runs unchanged.
    Return `[]` to skip an item the scorer can't judge (no reference answer).
    """

    def evaluator(
        *,
        input: dict[str, Any],
        output: RunResult,
        expected_output: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> dict[str, Any] | list[dict[str, Any]]:
        example = EvalCase(id="", inputs=input, reference_outputs=expected_output or {})
        try:
            verdict = scorer(run=output, example=example)
        except KeyError:
            return []
        return {"name": verdict.key, "value": verdict.score, "comment": verdict.comment}

    return evaluator


def make_task(model_id: str):
    """A Langfuse task: drive the graph with a real model on one dataset item (given)."""
    from langchain.chat_models import init_chat_model

    model = init_chat_model(model_id, api_key=get_settings().anthropic_api_key)

    def task(*, item: Any, **kwargs: Any) -> RunResult:
        return run(model, TOOLS, item.input["task"], max_steps=item.input.get("max_steps", 8))

    return task


if langfuse_configured() and get_settings().anthropic_api_key:
    from langfuse import Langfuse

    host, public_key, secret_key = require_langfuse()
    client = Langfuse(host=host, public_key=public_key, secret_key=secret_key)
    dataset = client.get_dataset("l13-agent-evals")  # uploaded by the L1302 lecture
    # TODO: launch the experiment over `dataset` and capture the result. Pass a `name`,
    #   a `run_name`, task=make_task("anthropic:claude-sonnet-4-6"), and
    #   evaluators=[as_evaluator(answer_correct), as_evaluator(no_runaway)].
    result = (
        ...
    )  # TODO: result = dataset.run_experiment(name=..., run_name=..., task=..., evaluators=...)
    client.flush()
    print("pushed a run — read the pass rate in Langfuse:", result.dataset_run_url)
else:
    print("Langfuse and/or ANTHROPIC_API_KEY not set — skipping the tooled push.")
    print("Configure both to run l13-agent-evals as a Langfuse experiment and read the rate.")

[↑ Back to top](#top)

## 6. Self-check

Compare against `L1305_lab_solutions.ipynb`. Done when:

- **Problem 1** the `no_runaway` pass rate for `hard_lookup` is strictly between 0 and 1 (a fraction like `3/5`), and the assert passes.
- **Problem 2** `compare(strong, weak)` flags `("recover", "answer_correct")` (and the lookup checks) as regressions; both asserts pass.
- **Problem 3** your written answer explains that one run is a coin flip on a non-deterministic agent, and that you'd report a **pass rate over K samples** instead.
- **Problem 4** (tooled) with both keys set, `dataset.run_experiment(...)` prints a run URL and the `l13-agent-evals` pass rates show up in Langfuse; offline it prints the skip note.

You can now measure pass rates and flag regressions across two versions — the ratchet you'll carry into every later agent. Next: [`L1306_lecture`](L1306_lecture.ipynb) asks what an eval run *costs*.

[↑ Back to top](#top)